# 03. Monotonic Stack: Next Greater / Smaller

A **monotonic stack** keeps its elements in sorted order (always increasing or always decreasing). When a new element breaks the order, we **pop** until the order is restored. The popped elements have just found their answer.

**Learning order:** REAL LIFE → IDEA → NEXT GREATER → CIRCULAR → DAILY TEMPERATURES → PREVIOUS SMALLER → STOCK SPAN → VARIANTS TABLE → INTERVIEW THINKING

## 1. Real life

People stand in a line facing right, each with a height. "Who is the **first taller person to my right**?" Walk from left to right keeping a stack of people still **waiting** for a taller person. When someone taller arrives, all shorter people on the stack have found their answer: pop them.

**Recognition clue:** "next greater", "next smaller", "how many days until warmer", "nearest to the left that is smaller".

## 2. Why O(n)?

Each element is **pushed once and popped once**. The inner `while` looks like nested loops, but the total number of pops is at most `n`.

Store **indexes** (not values) in the stack so you can compute distances and still read the value with `nums[i]`.

In [1]:
def next_greater(nums):
    res = [-1] * len(nums)
    stack = []                                # indexes, values decreasing
    for i, x in enumerate(nums):
        while stack and nums[stack[-1]] < x:
            res[stack.pop()] = x              # x is the next greater of the popped one
        stack.append(i)
    return res

print(next_greater([2, 1, 2, 4, 3]))   # [4, 2, 4, -1, -1]

[4, 2, 4, -1, -1]


## 3. Circular array (Next Greater Element II)

Treat the array as circular: walk the array **twice** (`i` from `0` to `2n-1`, use `i % n`). Push indexes only during the first pass.

In [2]:
def next_greater_circular(nums):
    n = len(nums)
    res = [-1] * n
    stack = []
    for i in range(2 * n):
        x = nums[i % n]
        while stack and nums[stack[-1]] < x:
            res[stack.pop()] = x
        if i < n:
            stack.append(i)
    return res

print(next_greater_circular([1, 2, 1]))   # [2, -1, 2]

def next_greater_element_i(nums1, nums2):           # nums1 is a subset of nums2
    nxt, stack = {}, []
    for x in nums2:
        while stack and stack[-1] < x:
            nxt[stack.pop()] = x
        stack.append(x)
    return [nxt.get(x, -1) for x in nums1]

print(next_greater_element_i([4, 1, 2], [1, 3, 4, 2]))   # [-1, 3, -1]

[2, -1, 2]
[-1, 3, -1]


## 4. Daily Temperatures

For each day, how many days until a **warmer** day? Same as next greater, but store `i - popped_index` (a distance).

In [3]:
def daily_temperatures(temps):
    res = [0] * len(temps)
    stack = []
    for i, t in enumerate(temps):
        while stack and temps[stack[-1]] < t:
            j = stack.pop()
            res[j] = i - j
        stack.append(i)
    return res

print(daily_temperatures([73, 74, 75, 71, 69, 72, 76, 73]))   # [1, 1, 4, 2, 1, 1, 0, 0]

[1, 1, 4, 2, 1, 1, 0, 0]


## 5. Previous Smaller element

Nearest smaller to the **left**. Keep an **increasing** stack. Before pushing `x`, pop everything `>= x` (they can never be the answer for anyone to the right if `x` is smaller or equal and closer). What remains on top is the answer for `x`.

In [4]:
def previous_smaller(nums):
    res, stack = [], []
    for x in nums:
        while stack and stack[-1] >= x:
            stack.pop()
        res.append(stack[-1] if stack else -1)
        stack.append(x)
    return res

print(previous_smaller([3, 7, 8, 4]))   # [-1, 3, 7, 3]

[-1, 3, 7, 3]


## 6. Online Stock Span

Span of today's price = number of consecutive days (including today) with price `<=` today's. Store `(price, span)`. When popping smaller-or-equal prices, **absorb their spans**.

In [5]:
class StockSpanner:
    def __init__(self):
        self.stack = []                         # (price, span)
    def next(self, price):
        span = 1
        while self.stack and self.stack[-1][0] <= price:
            span += self.stack.pop()[1]
        self.stack.append((price, span))
        return span

sp = StockSpanner()
print([sp.next(p) for p in [100, 80, 60, 70, 60, 75, 85]])   # [1, 1, 1, 2, 1, 4, 6]

[1, 1, 1, 2, 1, 4, 6]


## 7. Which stack for which question?

| Question | Scan direction | Pop while | Stack order |
|---|---|---|---|
| next **greater** to the right | left → right | `stack top < x` | decreasing |
| next **smaller** to the right | left → right | `stack top > x` | increasing |
| previous **greater** (to the left) | left → right | `stack top <= x`, then top is the answer | decreasing |
| previous **smaller** (to the left) | left → right | `stack top >= x`, then top is the answer | increasing |

For "next" the answer is given **when popping**. For "previous" the answer is the **top left after popping**.

## 8. Interview thinking

**Say:** "I maintain a stack of indexes whose answer is still unknown, in monotonic order. A new element resolves all smaller ones on top. Each index is pushed and popped once, so O(n)."

**Common mistakes:** storing values when you need indexes, `<` vs `<=` with duplicates, forgetting leftover stack elements (their answer is `-1`/`0`), not doubling the array for circular problems.

## Quick Revision

- Monotonic stack: pop while the new element breaks the order, popped items get their answer.
- Next greater: decreasing stack. Next smaller: increasing stack.
- Store indexes. O(n) because each index is pushed/popped once.
- Circular: loop `2n` times, `i % n`.
- Leftover elements = no answer.